# Granger-causality analysis

First import all necessary packages for Granger-causality analysis & forecasting

In [75]:
from utils.dip_utils import *
from utils.dip_visuals import *

import matplotlib.pyplot as plt
import pelz_datasets
import numpy as np

from statsmodels.tsa.stattools import adfuller

import warnings
warnings.filterwarnings('ignore')

In [76]:
plt.rcParams['font.size'] = 20
plt.rcParams['font.family'] = 'Arial'

# Data Preparation

In [77]:
output_prefix = 'data/outputs/plus1_log10_linear_imputation'
ts_data = pd.read_csv(f'{output_prefix}/lin_interpolated_ts_df.csv', index_col=0)
max_fixed_lag = 3

In [78]:
dvg_ts_data = ts_data[ts_data.columns[1:]].copy()
dvg_ts_data

## create random ts data

In [79]:
n_cols =  dvg_ts_data.shape[1]
np.random.seed(42)

fig,axs = plt.subplots(figsize=(12, 5), ncols=2)
ax1,ax2 = axs[0],axs[1]

# get per-time min and max (customize if you already have these)
min_vals = dvg_ts_data.min(axis=1)
max_vals = dvg_ts_data.max(axis=1)

# generate uniformly distributed samples between min and max
ts_data_random = pd.DataFrame(
    np.random.uniform(min_vals.values[:, None], max_vals.values[:, None], size=(len(dvg_ts_data), n_cols)),
    index=dvg_ts_data.index,
    columns=[f'rand_{i+1}' for i in range(n_cols)]
)

ts_data_random.sample(250, axis=1).plot(ax=ax1)
ax1.set_ylabel('NGS read counts (abs)')
ax1.legend().remove()
ax1.set_title('simulated data (bootstrapped min-max sampling)')

dvg_ts_data.plot(ax=ax2)
ax2.set_ylabel('NGS read counts (abs)')
ax2.legend().remove()
ax2.set_title('Measured data')
plt.tight_layout()
plt.show()


In [80]:
np.random.seed(42)
fig,axs = plt.subplots(figsize=(12, 5), ncols=2)
ax1,ax2 = axs[0],axs[1]
ts_data_shuffled = dvg_ts_data.apply(lambda x: np.random.permutation(x.values))
ts_data_shuffled.columns = [f'{col}_shuf' for col in ts_data_shuffled.columns]
ts_data_shuffled.plot(ax=ax1)
ax1.set_ylabel('NGS read counts (log10)')
ax1.legend().remove()
ax1.set_title('simulated data (shuffled per time series)')
dvg_ts_data.plot(ax=ax2)
ax2.set_ylabel('NGS read counts (log10)')
ax2.set_title('Measured data')
plt.legend().remove()

In [81]:
ts_data_random = pd.concat([ts_data['plaque_assay'], ts_data_random], axis=1)
ts_data_random

In [82]:
ts_data_shuffled = pd.concat([ts_data['plaque_assay'], ts_data_shuffled], axis=1)
ts_data_shuffled

In [93]:
ts_data_shuffled.to_csv(f'{output_prefix}/shuffled_lin_interpolated_ts_data.csv')
ts_data_random.to_csv(f'{output_prefix}/random_lin_interpolated_ts_data.csv')

In [97]:
ts_data_shuffled

dvg2nonzero_counts = {k:v for k,v in (ts_data_shuffled>0).sum().items()} | {k:v for k,v in (ts_data_random>0).sum().items()}
dvg2nonzero_counts

## Log10 transformation

In [83]:
log10_random_ts_data = ts_data_random.applymap(lambda x: np.log10(x+1))
log10_shuffled_ts_data = ts_data_shuffled.applymap(lambda x: np.log10(x+1))

## Stationarity

Time-series analysis relies on stationarity so that the models and predictions can be statistically relevant and to avoid the detection of spurious relationships.
Thus we need to pre process the imported raw data. I used two methods to obtain stationarity:
1. The iterative method
-> repeatedly form the difference of one value at index i with its preceeding value i-1 in the input array

2. The direct method 
-> directly determine which increment is needed to get a stationary time-series (determine x if np.diff(x) gives us a stationary dataframe)

### iterative method

In [84]:
def make_stationary(input_df, timepoint_idx=4, max_diff=10,
                    autolag=None, max_lag=None):
    """transform the input_df to a stationary time series by iterative differencing

    Args:
        input_df (pandas dataframe): input dataframe
        timepoint_idx (int, optional): index of the column where the time points start in the input_dataframe. Defaults to 4.
        max_diff (int, optional): maximum differencing level. How many times differencing should be done if no stationarity is reached yet.
                                    Defaults to 10.

    Returns:
        _type_: _description_
    """
    df = input_df.copy()
    list_stationary = []
    list_nonstationary = []
    diff_levels = []
    diff_nonstationary_list = []
    lags = []
    timepoint_columns = df.columns[timepoint_idx:]
    
    if max_lag is None:
        max_lag = int(np.sqrt(len(timepoint_columns)))
        
    if autolag is None:
        autolag = 'AIC'
        
    for i, row in df.iterrows():
        diff_level = 0
        p_value = 100
        ts_data = row[timepoint_columns]
        last_data = ts_data.copy()
        # Apply first differencing
        
        # Perform adfuller test
        try:
            with np.errstate(divide='ignore'):
                adf_result = adfuller(ts_data, 
                                    autolag=autolag,
                                    regression='ct', 
                                    maxlag=max_lag)
        except:
            list_nonstationary.append(row)
            continue
        p_value = adf_result[1]
        
        if p_value <= 0.05:
            # If the time series is stationary, add it to the stationary list
            row[ts_data.index] = ts_data.values
            list_stationary.append(row)
            lags.append(adf_result[2])
            diff_levels.append(diff_level)
        else:
            while diff_level < max_diff:
                diff_data = last_data.diff().dropna()
                diff_level += 1
                with np.errstate(divide='ignore'):
                    adf_result = adfuller(diff_data, autolag=autolag, maxlag=max_lag)
                p_value = adf_result[1]
                if p_value <= 0.05:
                    row[diff_data.index] = diff_data.values
                    list_stationary.append(row)
                    lags.append(adf_result[2])
                    diff_levels.append(diff_level)
                    break
                elif diff_level == max_diff:
                    list_nonstationary.append(row)
                    diff_nonstationary_list.append(diff_data)
                last_data = diff_data.copy()
        
        
    if len(list_stationary) > 0:
        df_stationary = pd.concat(list_stationary, axis=1).T
    else:
        df_stationary = pd.DataFrame()
    if len(list_nonstationary) > 0:
        df_nonstationary = pd.concat(list_nonstationary, axis=1).T
    else:
        df_nonstationary = pd.DataFrame()
    if len(diff_nonstationary_list) > 0:
        df_diff_nonstationary = pd.concat(diff_nonstationary_list, axis=1).T
    else:
        df_diff_nonstationary = pd.DataFrame()
    return df_stationary, df_nonstationary,df_diff_nonstationary, diff_levels, lags

import types

def make_stationary(input_df, max_diff=10, autolag=None, max_lag=None, regression="c"):
    """Transform each column (candidate) to a stationary series by iterative differencing.

    Args:
        input_df (pd.DataFrame): rows are timepoints; columns are candidates/variables.
        max_diff (int): maximum differencing level. Defaults to 10.
        autolag (str | None): lag selection for ADF. Defaults to 'AIC'.
        max_lag (int | None): max lags for ADF. Defaults to sqrt(n_timepoints), then clamped.
        regression (str): ADF regression trend ('c', 'ct', 'ctt', 'nc'). Defaults to 'c'.

    Returns:
        df_stationary (pd.DataFrame): final stationary series (may be differenced) for stationary columns.
        df_nonstationary (pd.DataFrame): original non-stationary columns.
        df_diff_nonstationary (pd.DataFrame): last differenced data for non-stationary columns (np.diff, padded).
        diff_levels (pd.Series): differencing level used per column (NaN for non-stationary/error).
        lags (pd.Series): ADF-selected lag per column (NaN for non-stationary/error).
        pvals (pd.Series): final ADF p-value per column (NaN on error).
        status (pd.Series): "stationary", "non-stationary", or "error".
    """
    df = input_df.copy()
    n_timepoints = len(df)

    if autolag is None:
        autolag = "AIC"
    if max_lag is None:
        max_lag = int(np.sqrt(max(n_timepoints, 1)))
    # Clamp to feasible range
    max_lag = max(1, min(max_lag, max(1, n_timepoints // 2 - 1)))

    stationary_data = {}
    nonstationary_cols = []
    diff_nonstationary_dict = {}
    diff_levels = {}
    lags = {}
    pvals = {}
    status = {}

    for col in df.columns:
        series = pd.to_numeric(df[col], errors="coerce")
        # Drop NaNs for testing but keep original index for reconstruction
        ts_data = series.dropna().to_numpy()
        if len(ts_data) < 3 or len(ts_data) <= max_lag:
            # Too short for ADF with this max_lag
            nonstationary_cols.append(col)
            diff_nonstationary_dict[col] = np.diff(series.to_numpy()) if len(series) > 1 else np.array([])
            diff_levels[col] = np.nan
            lags[col] = np.nan
            pvals[col] = np.nan
            status[col] = "error"
            continue

        diff_level = 0
        last_data = ts_data.copy()

        def run_adf(arr):
            with np.errstate(divide='ignore', invalid='ignore'):
                return adfuller(arr, autolag=autolag, regression=regression, maxlag=max_lag)

        # Initial ADF
        try:
            adf_result = run_adf(last_data)
            p_value = adf_result[1]
        except Exception:
            nonstationary_cols.append(col)
            diff_nonstationary_dict[col] = np.diff(series.to_numpy()) if len(series) > 1 else np.array([])
            diff_levels[col] = np.nan
            lags[col] = np.nan
            pvals[col] = np.nan
            status[col] = "error"
            continue

        if p_value <= 0.05:
            # Already stationary; keep original cleaned series, realign to original index
            stationary_series = series
            stationary_data[col] = stationary_series
            diff_levels[col] = diff_level
            lags[col] = adf_result[2]
            pvals[col] = p_value
            status[col] = "stationary"
            continue

        # Iterative differencing
        became_stationary = False
        while diff_level < max_diff:
            diff_data = np.diff(last_data)
            diff_level += 1
            # Need at least 3 points to run ADF reasonably
            if len(diff_data) < 3 or len(diff_data) <= max_lag:
                # Can't test further; mark non-stationary
                break
            try:
                adf_result = run_adf(diff_data)
                p_value = adf_result[1]
            except Exception:
                break

            if p_value <= 0.05:
                # Store differenced series aligned to original index (pad front with NaN)
                padded = np.concatenate(([np.nan] * diff_level, diff_data))
                # Trim/pad to original length
                padded = padded[:n_timepoints] if len(padded) >= n_timepoints else np.pad(
                    padded, (0, n_timepoints - len(padded)), constant_values=np.nan
                )
                stationary_series = pd.Series(padded, index=df.index)
                stationary_data[col] = stationary_series
                diff_levels[col] = diff_level
                lags[col] = adf_result[2]
                pvals[col] = p_value
                status[col] = "stationary"
                became_stationary = True
                break

            last_data = diff_data

        if not became_stationary:
            nonstationary_cols.append(col)
            # Keep the last differenced data for inspection
            diff_nonstationary_dict[col] = np.diff(series.to_numpy()) if len(series) > 1 else np.array([])
            diff_levels[col] = np.nan
            lags[col] = np.nan
            pvals[col] = np.nan
            status[col] = "non-stationary"

    # Assemble outputs
    df_stationary = pd.DataFrame(stationary_data, index=df.index) if stationary_data else pd.DataFrame(index=df.index)
    df_nonstationary = df[nonstationary_cols].copy() if nonstationary_cols else pd.DataFrame(index=df.index)

    if diff_nonstationary_dict:
        padded = {}
        for col, arr in diff_nonstationary_dict.items():
            # np.diff gives length n-1; pad with leading NaN to align to n
            padded_arr = np.concatenate(([np.nan], arr))
            if len(padded_arr) < n_timepoints:
                padded_arr = np.pad(padded_arr, (0, n_timepoints - len(padded_arr)), constant_values=np.nan)
            else:
                padded_arr = padded_arr[:n_timepoints]
            padded[col] = padded_arr
        df_diff_nonstationary = pd.DataFrame(padded, index=df.index)
    else:
        df_diff_nonstationary = pd.DataFrame(index=df.index)

    return types.SimpleNamespace(
        df_stationary=df_stationary,
        df_nonstationary=df_nonstationary,
        df_diff_nonstationary=df_diff_nonstationary,
        diff_levels=pd.Series(diff_levels),
        lags=pd.Series(lags),
        pvals=pd.Series(pvals),
        status=pd.Series(status),
    )


In [85]:
stationarity_results_random = make_stationary(log10_random_ts_data, max_diff=10, autolag='AIC', max_lag=None)
stationarity_results_shuffled = make_stationary(log10_shuffled_ts_data, max_diff=10, autolag='AIC', max_lag=None)


In [86]:
random_ts_df = stationarity_results_random.df_stationary
shuffled_ts_df = stationarity_results_shuffled.df_stationary

# Granger-causality test

In [87]:
# importing the granger causality test from statsmodels
from statsmodels.tsa.stattools import grangercausalitytests

# assigning the string 'ssr_chi2test' to the variable 'test'
test = 'ssr_chi2test'
err_dips = {}

In [88]:
def granger_causation_matrix_fixed_lag(data, variables, cultivation_values=[],test='ssr_chi2test', plot=False,fixlag=3, maxlag=3, debug=False):
    test_results = {}
    err_dips = {}
    max_lags = {}
    # creating a dataframe with the same dimensions as number of variables entered, assigned to the variable 'X_train'
    X_train = pd.DataFrame(np.nan, index=variables, columns=variables)
    
    # loops through the columns and the indexes
    for c in X_train.columns:
        for r in X_train.index:
            #skip if we're not looking at any cultivation variable
            if r == c:
                continue
            go_on = False
            if c in cultivation_values:
                go_on = True
            if r in cultivation_values:
                go_on = True
            
            if go_on == False:
                continue
            # conducts a granger causality test on a variable row and column using the 'maxlag' variable; assigns to 
            # variable test_result
            try:
                test_result = grangercausalitytests(data[[r, c]], maxlag=[maxlag], verbose=False)
            except Exception as e:
                if debug:
                  print(f'Error on {r} and {c}, {data[[r, c]]}: {e}')
                err_dips[(r,c)] = e
                X_train.loc[r, c] = None
                max_lags[(r,c)] = None
                continue
            test_results[(r,c)] = test_result    
            # locates the test result in the tuple 'test_result' and rounds the number by 4 digits; assigns to 'p_values'
            p_value = round(test_result[fixlag][0][test][1], 4)
            max_lags[(r,c)] = fixlag
            X_train.loc[r, c] = p_value
            
    # rename the row and column names based on the relationship
    X_train.columns = [var + '_x' for var in variables]
    X_train.index = [var + '_y' for var in variables]
    return X_train, test_results, max_lags, err_dips

In [89]:
from statsmodels.stats.multitest import multipletests

import numpy as np
from statsmodels.stats.multitest import multipletests

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

def correct_p_values_bh_separately(matrix, cult_columns=5, alpha=0.05, debug=False):
    """
    Applies Benjamini-Hochberg correction cultivation-column-wise:
    1. DVGs -> each Virus Concentration column (row i, DVG columns to the right)
    2. Each Virus Concentration column -> DVGs (DVG rows, column i)

    Parameters:
        matrix (pd.DataFrame): A pandas DataFrame where each cell contains a p-value.
        cult_columns (int): Number of columns corresponding to virus concentration variables.
        alpha (float): significance level for BH.
        debug (bool): if True, print and display slices being corrected.

    Returns:
        pd.DataFrame: Same shape with BH-adjusted p-values.
    """
    corrected_matrix = matrix.copy()

    for col_i in range(cult_columns):
        col_name = matrix.columns[col_i].replace("_x", "")

        # DVG -> col_i (row col_i, DVG columns to the right)
        dvg_to_col = matrix.iloc[col_i, cult_columns:].values
        if debug:
            print(f"\n[DVG -> {col_name}] slice BEFORE correction:")
            display(matrix.iloc[col_i:col_i+1, cult_columns:])
        mask = ~np.isnan(dvg_to_col)
        if mask.any():
            _, corrected_vals, _, _ = multipletests(dvg_to_col[mask], alpha=alpha, method="fdr_bh")
            tmp = dvg_to_col.copy()
            tmp[mask] = corrected_vals
            corrected_matrix.iloc[col_i, cult_columns:] = tmp
            if debug:
                print(f"[DVG -> {col_name}] slice AFTER correction:")
                display(corrected_matrix.iloc[col_i:col_i+1, cult_columns:])

        # col_i -> DVGs (all DVG rows below, column col_i)
        col_to_dvg = matrix.iloc[cult_columns:, col_i].values
        if debug:
            print(f"\n[{col_name} -> DVGs] slice BEFORE correction:")
            display(matrix.iloc[cult_columns:, col_i:col_i+1])
        mask = ~np.isnan(col_to_dvg)
        if mask.any():
            _, corrected_vals, _, _ = multipletests(col_to_dvg[mask], alpha=alpha, method="fdr_bh")
            tmp = col_to_dvg.copy()
            tmp[mask] = corrected_vals
            corrected_matrix.iloc[cult_columns:, col_i] = tmp
            if debug:
                print(f"[{col_name} -> DVGs] slice AFTER correction:")
                display(corrected_matrix.iloc[cult_columns:, col_i:col_i+1])
    return corrected_matrix

def _bh_critical_pval(pvals, alpha=0.05, debug=False):
    """Return the BH critical p-value for a 1D array (ignoring NaNs)."""
    p = np.sort(np.asarray(pvals)[~np.isnan(pvals)])
    m = p.size
    if m == 0:
        if debug:
            print("No valid p-values found.")
        return None
    thresholds = alpha * (np.arange(1, m+1) / m)
    hits = np.where(p <= thresholds)[0]
    if debug:
      # create df
      tmp_df = pd.DataFrame({'p-values': p, 'thresholds': thresholds})
      tmp_df['hits'] = np.where(p <= thresholds, 'hit', 'miss')
      display(tmp_df)
    if hits.size == 0:
      if debug:
          print("No hits found.")
      return None
    kmax = hits.max()
    if debug:
      print(f"Critical p-value for BH correction at rank {kmax}: {p[kmax]}")
    return p[kmax]   # the largest p-value that still passes

def calculate_critical_pval_bh(matrix, cult_columns=5, alpha=0.05, debug=False):
    """
    For each cultivation column i (0..cult_columns-1), return:
      - 'dvg_to_<colname>': BH critical p for DVGs -> <colname>
        (row i, DVG columns to the right)
      - '<colname>_to_dvg': BH critical p for <colname> -> DVGs
        (DVG rows below, column i)
    """
    crit_dct = {}
    for i, col_name in enumerate(matrix.columns[:cult_columns]):
        clean = col_name.replace('_x', '')

        # DVG -> col_i  (row i, DVG columns)
        dvg_to_col_slice = matrix.iloc[i, cult_columns:]
        if debug:
            print(f"\n[DVG -> {clean}] slice:")
            display(dvg_to_col_slice.to_frame().T)
        dvg_to_col = dvg_to_col_slice.values
        crit_dct[f"dvg_to_{clean}"] = _bh_critical_pval(dvg_to_col, alpha, debug)

        # col_i -> DVGs  (DVG rows, column i)
        col_to_dvg_slice = matrix.iloc[cult_columns:, i]
        if debug:
            print(f"\n[{clean} -> DVGs] slice:")
            display(col_to_dvg_slice.to_frame())
        col_to_dvg = col_to_dvg_slice.values
        crit_dct[f"{clean}_to_dvg"] = _bh_critical_pval(col_to_dvg, alpha, debug)

    return crit_dct

## Make Granger-causality matrix

How to read the matrix: column X Granger-causes row Y
i.e. X improves the forecasting performance of Y if included in an OLS model

In [90]:
import pickle
import os
output_prefix = 'data/outputs/plus1_log10_random_shuffled'
if not os.path.exists(output_prefix):
  os.makedirs(output_prefix)
  
gc_matrix = {}
test_results = {}
gc_max_lag = {}
errors = {}

corrected_gc_matrix = {}
bh_critical_p_vals = {}

In [91]:
stationary_ts_df = pd.concat([random_ts_df, shuffled_ts_df[shuffled_ts_df.columns[1:]]], axis=1)
stationary_ts_df

In [94]:
output_prefix

In [92]:
stationary_ts_df.to_csv(f'{output_prefix}/stationary_ts_df.csv')

In [98]:
for fix_lag in range(1,max_fixed_lag):
    print(f'Calculating granger causality matrix for fixed lag {fix_lag}...')
    gc_matrix[fix_lag], test_results[fix_lag], gc_max_lag[fix_lag], errors[fix_lag] = granger_causation_matrix_fixed_lag(
        stationary_ts_df,
        variables=stationary_ts_df.columns.tolist(),
        cultivation_values=['plaque_assay'],
        fixlag=fix_lag,
        maxlag=fix_lag
    )
    
    corrected_gc_matrix[fix_lag] = correct_p_values_bh_separately(
        gc_matrix[fix_lag],
        cult_columns=1,
        alpha=0.05,
        debug=False
    )
    
    bh_critical_p_vals[fix_lag] = calculate_critical_pval_bh(
        gc_matrix[fix_lag],
        cult_columns=1,
        alpha=0.05,
        debug=False
    )

In [99]:
print("critical p-values with BH correction:")
bh_critical_p_vals

In [100]:
import matplotlib.pyplot as plt
import math

# Calculate grid dimensions (e.g., 3 columns)
n_plots = len(gc_matrix)
cols = 6
rows = math.ceil(n_plots / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 2 * rows), constrained_layout=True)
axes = axes.flatten()  # Flatten to iterate easily

# set font size
plt.rcParams.update({'font.size': 8})

for i, fixlag in enumerate(range(1, n_plots + 1)):
    ax = axes[i]
    
    # Plotting the histogram on the specific axis
    # Note: using fixlag instead of the hardcoded 3 from your snippet
    gc_matrix[fixlag].iloc[0].hist(bins=50, ax=ax)
    
    ax.set_title(f'lag={fixlag}')
    ax.set_xlabel('p-value')
    ax.set_ylabel('Frequency')

# Hide any empty subplots if the grid is larger than the number of lags
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

# Set the main title for the entire figure
plt.suptitle("Histogram of p-values for granger causality DVG->PFU", fontsize=10)
plt.show()

fig_pfu_to_dvg, axes_pfu_to_dvg = plt.subplots(rows, cols, figsize=(15, 2 * rows), constrained_layout=True)
axes_pfu_to_dvg = axes_pfu_to_dvg.flatten()  # Flatten to iterate easily
for i, fixlag in enumerate(range(1, n_plots + 1)):
    ax = axes_pfu_to_dvg[i]
    
    # Plotting the histogram on the specific axis
    # Note: using fixlag instead of the hardcoded 3 from your snippet
    gc_matrix[fixlag].iloc[1:,0].hist(bins=50, ax=ax)
    
    ax.set_title(f'lag={fixlag}')
    ax.set_xlabel('p-value')
    ax.set_ylabel('Frequency')
# Hide any empty subplots if the grid is larger than the number of lags
for j in range(i + 1, len(axes_pfu_to_dvg)):
    axes_pfu_to_dvg[j].axis('off')
# Set the main title for the entire figure
plt.suptitle("Histogram of p-values for granger causality PFU->DVG", fontsize=10)
plt.show()

## Summarize

In [101]:
def classify_granger_dips(gc_matrix, cultivation_vals):
    granger_caused_dips = {}
    granger_not_caused_dips = {}
    granger_causing_dips = {}
    granger_not_causing_dips = {}
    granger_bi_dips = {}
    non_related_dips = {}
    for cultivation_val in cultivation_vals:
        granger_caused_dips[cultivation_val] = [dip.replace('_y', '') for dip in gc_matrix[gc_matrix[cultivation_val + '_x'] <= 0.05].index]
        granger_not_caused_dips[cultivation_val] = [dip.replace('_y', '') for dip in gc_matrix[gc_matrix[cultivation_val + '_x'] > 0.05].index]
        tmp_y = gc_matrix.loc[cultivation_val + '_y']
        granger_causing_dips[cultivation_val] = [dip.replace('_x', '') for dip in tmp_y[tmp_y <= 0.05].index]
        granger_not_causing_dips[cultivation_val] = [dip.replace('_x', '') for dip in tmp_y[tmp_y > 0.05].index]
        
        granger_bi_dips[cultivation_val] = list(set(granger_causing_dips[cultivation_val]).intersection(set(granger_caused_dips[cultivation_val])))
        non_related_dips[cultivation_val] = list(set(granger_not_caused_dips[cultivation_val]).intersection(set(granger_not_causing_dips[cultivation_val])))
    return granger_caused_dips, granger_causing_dips, granger_bi_dips, non_related_dips
  
def classify_granger_dips_bh(gc_matrix, cultivation_vals, bh_critical_pvals):    
    granger_caused_dips = {}
    granger_not_caused_dips = {}
    granger_causing_dips = {}
    granger_not_causing_dips = {}
    granger_bi_dips = {}
    non_related_dips = {}
    for cultivation_val in cultivation_vals:
        critical_causing_pval = bh_critical_pvals[f"dvg_to_{cultivation_val}"]
        critical_caused_pval = bh_critical_pvals[f"{cultivation_val}_to_dvg"]
        granger_caused_dips[cultivation_val] = [dip.replace('_y', '') for dip in gc_matrix[gc_matrix[cultivation_val + '_x'] <= critical_caused_pval].index]
        granger_not_caused_dips[cultivation_val] = [dip.replace('_y', '') for dip in gc_matrix[gc_matrix[cultivation_val + '_x'] > critical_caused_pval].index]
        tmp_y = gc_matrix.loc[cultivation_val + '_y']
        granger_causing_dips[cultivation_val] = [dip.replace('_x', '') for dip in tmp_y[tmp_y <= critical_causing_pval].index]
        granger_not_causing_dips[cultivation_val] = [dip.replace('_x', '') for dip in tmp_y[tmp_y > critical_causing_pval].index]
        
        granger_bi_dips[cultivation_val] = list(set(granger_causing_dips[cultivation_val]).intersection(set(granger_caused_dips[cultivation_val])))
        non_related_dips[cultivation_val] = list(set(granger_not_caused_dips[cultivation_val]).intersection(set(granger_not_causing_dips[cultivation_val])))
    return granger_caused_dips, granger_causing_dips, granger_bi_dips, non_related_dips
        

In [102]:
granger_caused_dips = {}
granger_causing_dips = {}
granger_bi_dips = {}
non_related_dips = {}

granger_caused_dips_bh = {}
granger_causing_dips_bh = {}
granger_bi_dips_bh = {}
non_related_dips_bh = {}

granger_caused_dips_corrected = {}
granger_causing_dips_corrected = {}
granger_bi_dips_corrected = {}
non_related_dips_corrected = {}

In [103]:
for fix_lag in range(1,max_fixed_lag):
  granger_caused_dips[fix_lag], granger_causing_dips[fix_lag], granger_bi_dips[fix_lag], non_related_dips[fix_lag] = classify_granger_dips(
      gc_matrix[fix_lag],
      cultivation_vals=['plaque_assay']
  )
  
  granger_caused_dips_bh[fix_lag], granger_causing_dips_bh[fix_lag], granger_bi_dips_bh[fix_lag], non_related_dips_bh[fix_lag] = classify_granger_dips_bh(
      gc_matrix[fix_lag],
      cultivation_vals=['plaque_assay'],
      bh_critical_pvals=bh_critical_p_vals[fix_lag]
  )
  
  granger_caused_dips_corrected[fix_lag], granger_causing_dips_corrected[fix_lag], granger_bi_dips_corrected[fix_lag], non_related_dips_corrected[fix_lag] = classify_granger_dips(
      corrected_gc_matrix[fix_lag],
      cultivation_vals=['plaque_assay']
  )


In [104]:
def create_summary_df(gc_test_results, 
                      gc_max_lag,
                      granger_caused_dips, granger_causing_dips, granger_bi_dips, non_related_dips,
                      diff_level_dct,
                      readcounts_data,
                      time_series_data,
                      output_file, 
                      stat_tests=['ssr_ftest', 'ssr_chi2test', 'lrtest'],
                      cultivation_values=['vrna', 'tcid50', 'plaque_assay', 'log_ha', 'ha_titer']):
    
    # Collect all unique DIPs that were tested (including non-related)
    all_tested_dips = set(
        dip for dips in granger_caused_dips.values() for dip in dips
    ).union(
        dip for dips in granger_causing_dips.values() for dip in dips
    ).union(
        dip for dips in non_related_dips.values() for dip in dips
    )

    # Initialize summary dataframe with filtered keys
    summary_df = readcounts_data[readcounts_data['key'].isin(all_tested_dips)][['key']].copy()
    
    # Assign Granger causality labels
    for cultivation_value in cultivation_values:
        summary_df[cultivation_value + '_granger_label'] = summary_df['key'].apply(
            lambda dip: (
                'bi-directional' if dip in granger_bi_dips.get(cultivation_value, [])
                else 'causing' if dip in granger_causing_dips.get(cultivation_value, [])
                else 'caused' if dip in granger_caused_dips.get(cultivation_value, [])
                else 'non-related' if dip in non_related_dips.get(cultivation_value, []) 
                else None  # DIPs that were never processed
            )
        )
    
    # Store max_diff level
    summary_df['max_diff'] = summary_df['key'].map(diff_level_dct).fillna(0)
    
    # Function to extract Granger test statistics (Unchanged)
    def extract_granger_stats(key, cultivation_value, direction):
        """Extracts Granger test statistics, lags, and correlation values."""
        tr_key = (cultivation_value, key) if direction == 'causing' else (key, cultivation_value)
        
        if tr_key not in gc_test_results or tr_key not in gc_max_lag:
            return None, None, None, [None] * len(stat_tests)
        
        tr = gc_test_results[tr_key]
        max_lag = gc_max_lag[tr_key]
        
        # Extract p-values for each statistical test
        
        stat_pvals = [tr[max_lag][0][test][0] for test in stat_tests]
        
        # Compute cross-correlation
        cross_corr_value = time_series_data[key].corr(
            time_series_data[cultivation_value].shift(-max_lag)  # Always shift as "causing"
        )
        
        return max_lag, cross_corr_value, stat_pvals
    
    # Process each cultivation variable
    for cultivation_value in cultivation_values:
        max_lags, cross_corr_values, significant_pvals_lags = [], [], []
        stat_tests_dct = {test: [] for test in stat_tests}
        
        for key in summary_df['key']:
            granger_label = summary_df.loc[summary_df['key'] == key, cultivation_value + '_granger_label'].values[0]
            
            # Extract Granger stats for all tested DIPs, including "non-related"
            if granger_label is not None:
                direction = 'causing'  # Always using "causing" for now
                
                max_lag, cross_corr, stat_pvals = extract_granger_stats(key, cultivation_value, direction)
                
                max_lags.append(max_lag)
                cross_corr_values.append(cross_corr)
                
                for i, test in enumerate(stat_tests):
                    stat_tests_dct[test].append(stat_pvals[i])
            else:
                # Skip unprocessed dips
                max_lags.append(None)
                cross_corr_values.append(None)
                for test in stat_tests:
                    stat_tests_dct[test].append(None)
        
        # Assign extracted values to dataframe
        summary_df[cultivation_value + '_max_lag'] = max_lags
        summary_df[cultivation_value + '_cross_corr'] = cross_corr_values
        
        for test in stat_tests:
            summary_df[cultivation_value + '_' + test] = stat_tests_dct[test]
        
        summary_df[cultivation_value + '_significant_pvals'] = summary_df[
            [cultivation_value + '_' + test for test in stat_tests]
        ].lt(0.05).sum(axis=1)
    
    # Compute Granger score
    granger_score_columns = [
        col for cultivation_value in cultivation_values 
        for col in [cultivation_value + '_significant_pvals']
    ]
    summary_df['granger_score'] = summary_df[granger_score_columns].sum(axis=1) - summary_df['max_diff']
    
    # Sort by Granger score and max_diff
    summary_df = summary_df.sort_values(by=['granger_score', 'max_diff'], ascending=[False, True])
    
    # Save to CSV
    summary_df.to_csv(output_file, index=False)

    return summary_df

In [105]:
read_counts_df = pd.DataFrame(columns=['key'])
read_counts_df = pd.concat([read_counts_df, stationary_ts_df.T[1:].reset_index().rename(columns={'index': 'key'})], ignore_index=True)
read_counts_df


In [106]:
summary_df = {}
summary_df_corrected = {}
summary_df_bh = {}


In [107]:
for fixlag in range(1, max_fixed_lag):
  summary_df[fixlag] = create_summary_df(
      gc_test_results=test_results[fixlag],
      gc_max_lag=gc_max_lag[fixlag],
      granger_causing_dips=granger_causing_dips[fixlag],
      granger_caused_dips=granger_caused_dips[fixlag],
      granger_bi_dips=granger_bi_dips[fixlag],
      non_related_dips=non_related_dips[fixlag],
      diff_level_dct=stationarity_results_random.diff_levels.to_dict() | stationarity_results_shuffled.diff_levels.to_dict(),
      readcounts_data=read_counts_df,
      time_series_data=stationary_ts_df,
      output_file=f'{output_prefix}/gc_summary_df_lag{fixlag}.csv',
      stat_tests=['ssr_ftest', 'ssr_chi2test', 'lrtest'],
      cultivation_values=['plaque_assay']   
  )
  
  summary_df_corrected[fixlag] = create_summary_df(
      gc_test_results=test_results[fixlag],
      gc_max_lag=gc_max_lag[fixlag],
      granger_causing_dips=granger_causing_dips_corrected[fixlag],
      granger_caused_dips=granger_caused_dips_corrected[fixlag],
      granger_bi_dips=granger_bi_dips_corrected[fixlag],
      non_related_dips=non_related_dips_corrected[fixlag],
      diff_level_dct=stationarity_results_random.diff_levels.to_dict() | stationarity_results_shuffled.diff_levels.to_dict(),
      readcounts_data=read_counts_df,
      time_series_data=stationary_ts_df,
      output_file=f'{output_prefix}/gc_summary_df_corrected_lag{fixlag}.csv',
      stat_tests=['ssr_ftest', 'ssr_chi2test', 'lrtest'],
      cultivation_values=['plaque_assay']   
  )
  
  summary_df_bh[fixlag] = create_summary_df(
      gc_test_results=test_results[fixlag],
      gc_max_lag=gc_max_lag[fixlag],
      granger_causing_dips=granger_causing_dips_bh[fixlag],
      granger_caused_dips=granger_caused_dips_bh[fixlag],
      granger_bi_dips=granger_bi_dips_bh[fixlag],
      non_related_dips=non_related_dips_bh[fixlag],
      diff_level_dct=stationarity_results_random.diff_levels.to_dict() | stationarity_results_shuffled.diff_levels.to_dict(),
      readcounts_data=read_counts_df,
      time_series_data=stationary_ts_df,
      output_file=f'{output_prefix}/gc_summary_df_bh_lag{fixlag}.csv',
      stat_tests=['ssr_ftest', 'ssr_chi2test', 'lrtest'],
      cultivation_values=['plaque_assay']   
  )

In [108]:
stationary_df = stationary_ts_df.T.copy()
stationary_df

In [109]:
with_gc_labels = pd.DataFrame()
with_gc_labels.index = stationary_df.index


In [110]:
key2label = {}
for fixlag in range(1, max_fixed_lag):
  for cultivation_value in ['plaque_assay']:
    for idx, row in summary_df[fixlag].iterrows():
        key = row['key']
        label = row[cultivation_value + '_granger_label']
        if key not in key2label:
            key2label[key] = {}
        key2label[key][f'lag{fixlag}_{cultivation_value}'] = label
        
        with_gc_labels.loc[key, f'lag{fixlag}_{cultivation_value}_granger_label'] = label

with_gc_labels = pd.concat([with_gc_labels, stationary_df], axis=1)


In [111]:
with_gc_labels

In [112]:
# drop rows if all cols ['pfu_label_lag1', 'pfu_label_lag2', 'pfu_label_lag3', 'pfu_label_lag4'] are NaN
with_gc_labels = with_gc_labels.dropna(subset=with_gc_labels.columns[:max_fixed_lag-1], how='all')
with_gc_labels.to_csv(f'{output_prefix}/dvg_time_series_with_gc_labels.csv')


In [113]:
with_gc_labels

# Plot fitted models

## Functions

### Performance metrics

In [114]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from tslearn.metrics import dtw, dtw_path

from scipy.signal import argrelextrema
from scipy.stats import ttest_ind, mannwhitneyu, shapiro, pearsonr


In [115]:
def nrsme(y_true, y_pred):
    return root_mean_squared_error(y_true, y_pred, squared=False) / (np.max(y_true) - np.min(y_true))

def mase(y_true, y_pred):
    # Calculate naive forecast (shifted by one time step)
    naive_forecast = np.roll(y_true, shift=1)
    # Remove the first element to align with y_true[1:]
    naive_forecast = naive_forecast[1:]
    
    # Calculate MAE of the model
    mae_model = mean_absolute_error(y_true, y_pred)
    # Calculate MAE of the naive forecast
    mae_naive = mean_absolute_error(y_true[1:], naive_forecast)
    
    return mae_model / mae_naive

def find_local_extrema(y, order=1):
    """
    Find indices of local minima and maxima in the array.
    
    Parameters:
    - y: Array of values.
    - order: How many points on each side to use for the comparison to consider a point a local minimum or maximum.
    
    Returns:
    - minima_indices: Indices of local minima.
    - maxima_indices: Indices of local maxima.
    """
    minima_indices = argrelextrema(y, np.less, order=order)[0]
    maxima_indices = argrelextrema(y, np.greater, order=order)[0]
    
    # Check the first and last points manually
    if len(y) > 1:
        if y[0] < y[1]:
            minima_indices = np.insert(minima_indices, 0, 0)
        if y[0] > y[1]:
            maxima_indices = np.insert(maxima_indices, 0, 0)
        if y[-1] < y[-2]:
            minima_indices = np.append(minima_indices, len(y) - 1)
        if y[-1] > y[-2]:
            maxima_indices = np.append(maxima_indices, len(y) - 1)
    
    return minima_indices, maxima_indices

def calculate_weighted_mae(y_true, y_pred, order=1, weight_factor=3):
    """
    Calculate the Weighted Mean Absolute Error (WMAE) with higher weights on local minima and maxima.
    
    Parameters:
    - y_true: Array of true values.
    - y_pred: Array of predicted values.
    - order: Order for finding local minima and maxima.
    - weight_factor: Weight assigned to local minima and maxima.
    
    Returns:
    - wmae: Weighted Mean Absolute Error.
    """
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    # Find local minima and maxima
    minima_indices, maxima_indices = find_local_extrema(y_true_arr, order)
    
    # Initialize weights to 1
    weights = np.ones_like(y_true_arr)
    
    # Assign higher weights to local minima and maxima
    weights[minima_indices] = weight_factor
    weights[maxima_indices] = weight_factor
    
    # Calculate weighted MAE
    wmae = np.sum(weights * np.abs(y_true_arr - y_pred_arr) / np.sum(weights))
    
    return wmae

def calculate_weighted_mape(y_true, y_pred, order=1, weight_factor=3):
    """
    Calculate the Weighted Mean Absolute Percentage Error (WMAPE) with higher weights on local minima and maxima.
    
    Parameters:
    - y_true: Array of true values.
    - y_pred: Array of predicted values.
    - order: Order for finding local minima and maxima.
    - weight_factor: Weight assigned to local minima and maxima.
    
    Returns:
    - wmape: Weighted Mean Absolute Percentage Error.
    """
    y_true_arr = np.array(y_true)
    y_pred_arr = np.array(y_pred)
    # Find local minima and maxima
    minima_indices, maxima_indices = find_local_extrema(y_true_arr, order)
    
    # Initialize weights to 1
    weights = np.ones_like(y_true_arr)
    
    # Assign higher weights to local minima and maxima
    weights[minima_indices] = weight_factor
    weights[maxima_indices] = weight_factor
    
    # Calculate WMAPE
    # To avoid division by zero, we add a small epsilon value to y_true
    epsilon = 1e-10
    wmape = np.sum(weights * np.abs((y_true_arr - y_pred_arr) / (y_true_arr + epsilon))) / np.sum(weights)
    
    return wmape

def dtw_pearson(y_true, y_pred):
    """
    Calculate the Dynamic Time Warping (DTW) aligned Pearson correlation coefficient between two time series.
    
    Parameters:
    - y_true: Array of true values.
    - y_pred: Array of predicted values.
    
    Returns:
    - pearson_corr: The Pearson correlation coefficient between the aligned (warped) y_true and y_pred.
    """
    y_true = np.array(y_true).reshape(-1, 1)
    y_pred = np.array(y_pred).reshape(-1, 1)
    path, dtw_distance = dtw_path(y_true, y_pred)
    aligned_true = np.array([y_true[i] for i, j in path]).flatten()
    aligned_pred = np.array([y_pred[j] for i, j in path]).flatten()
    pearson_corr = pearsonr(aligned_true, aligned_pred)[0]
    return pearson_corr

def dtw_mape(y_true, y_pred, epsilon=1e-6):
    """
    Calculate the DTW-aligned Mean Absolute Percentage Error (MAPE) between two time series.
    
    Parameters:
    - y_true: Array of true values.
    - y_pred: Array of predicted values.
    - epsilon: Small constant to avoid division by zero in MAPE calculation.
    
    Returns:
    - dtw_distance: The DTW distance between y_true and y_pred.
    - dtw_aligned_mape: The MAPE calculated on the DTW-aligned series.
    """
    y_true = np.array(y_true).reshape(-1, 1)
    y_pred = np.array(y_pred).reshape(-1, 1)
    
    # Calculate DTW path and distance
    path, dtw_distance = dtw_path(y_true, y_pred)
    
    # Extract aligned series
    aligned_true = np.array([y_true[i] for i, j in path]).flatten()
    aligned_pred = np.array([y_pred[j] for i, j in path]).flatten()
    
    # Calculate MAPE on aligned series
    dtw_aligned_mape = np.mean(np.abs((aligned_true - aligned_pred) / (aligned_true + epsilon))) * 100
    
    return dtw_aligned_mape



### Plot single fits

In [116]:
granger_label_color_map = {
  'non-related': 'gray',
  'causing': 'tab:blue',
  'caused': 'tab:red',
  'bi-directional': 'tab:purple'
}

def run_granger_prediction(
    summary_df,
    gc_max_lag,
    gc_test_results,
    plot_dips,
    log_long_data,
    restricted_pred_index=None,
    include_ssr=False,
    figsize=(8,6),
    ylabel='log10(PFU)',
    xlabel='Time post infection (days)',
    cultivation_value_label='plaque_assay',
    retransform=False,
    yscale='linear'
):
    # Initialize data structures
    columns = ['key', 'granger_label', 'optimal_lag', 'diff_level', 'ssr_chi2test_pval', 'significant_pvals',
                'mape', 'mae', 'rmse', 'nrsme', 'mse', 'dtw', 'wmae', 'wmape', 'relative_mape',
                'relative_rmse', 'relative_mae', 'relative_wmae', 'relative_wmape', 'relative_dtw',
                'relative_mse', 'dtw_aligned_mape', 'ssr', 'weighted_ssr', 'pred_data']

    gc_prediction_summary_tmp = pd.DataFrame(columns=columns)
    ref_ols_performance_dct_tmp = {}
    full_preds = {}

    # Prepare dataframe
    summary_df_cp = summary_df.copy()
    summary_df_cp[f'{cultivation_value_label}_granger_label'] = summary_df_cp[f'{cultivation_value_label}_granger_label'].fillna('non-related')

    if retransform:
      # retransform all time series data
      log_long_data = np.power(10, log_long_data)
    restricted_pred_done = False
    for idx, row in summary_df_cp.iterrows():
        dip = row.key
        opt_lag = gc_max_lag[(cultivation_value_label, dip)]
        full_pred = gc_test_results[cultivation_value_label, dip][opt_lag][1][1].predict()
        if retransform:
          full_pred = np.power(10, full_pred)
          
        if not restricted_pred_done:
          restricted_pred_index = idx if restricted_pred_index is None else restricted_pred_index
          restricted_pred = gc_test_results[cultivation_value_label, dip][opt_lag][1][0].predict()
          if retransform:
            # Retransform predictions if necessary
            restricted_pred = np.power(10, restricted_pred)
          restricted_pred_done = True
                
        full_preds[dip] = full_pred
        actual_value = log_long_data[cultivation_value_label]
        actual_value_comp = actual_value.iloc[opt_lag:]

        dpis = log_long_data.index.values

        metrics = {
              'mape': mean_absolute_percentage_error(actual_value_comp, full_pred),
              'mae': mean_absolute_error(actual_value_comp, full_pred),
              'mse': root_mean_squared_error(actual_value_comp, full_pred)**2
          }

        metrics['rmse'] = np.sqrt(metrics['mse'])
        metrics['nrsme'] = metrics['rmse'] / np.mean(actual_value)
        metrics['dtw'] = dtw(actual_value_comp, full_pred)
        metrics['wmae'] = calculate_weighted_mae(actual_value_comp, full_pred)
        metrics['wmape'] = calculate_weighted_mape(actual_value_comp, full_pred)

        # Relative errors
        rel_metrics = {
            'relative_mape': metrics['mape'] / mean_absolute_percentage_error(actual_value_comp, restricted_pred),
            'relative_rmse': metrics['rmse'] / root_mean_squared_error(actual_value_comp, restricted_pred),
            'relative_mae': metrics['mae'] / mean_absolute_error(actual_value_comp, restricted_pred),
            'relative_wmae': metrics['wmae'] / calculate_weighted_mae(actual_value_comp, restricted_pred),
            'relative_wmape': metrics['wmape'] / calculate_weighted_mape(actual_value_comp, restricted_pred),
            'relative_dtw': metrics['dtw'] / dtw(actual_value_comp, restricted_pred),
            'relative_mse': metrics['mse'] / root_mean_squared_error(actual_value_comp, restricted_pred)**2
        }

        ssr_value = np.sum((actual_value_comp - full_pred) ** 2) if include_ssr else None
        weighted_ssr = np.sum(((actual_value_comp - full_pred)/max(actual_value_comp)) ** 2) / len(full_pred) if include_ssr else None
        
        gc_prediction_summary_tmp.loc[len(gc_prediction_summary_tmp)] = [
            dip, row[f'{cultivation_value_label}_granger_label'], opt_lag, row['max_diff'], row[f'{cultivation_value_label}_ssr_chi2test'],
            row[f'{cultivation_value_label}_significant_pvals'], metrics['mape'], metrics['mae'], metrics['rmse'],
            metrics['nrsme'], metrics['mse'], metrics['dtw'], metrics['wmae'], metrics['wmape'],
            rel_metrics['relative_mape'], rel_metrics['relative_rmse'], rel_metrics['relative_mae'],
            rel_metrics['relative_wmae'], rel_metrics['relative_wmape'], rel_metrics['relative_dtw'],
            rel_metrics['relative_mse'], dtw_mape(actual_value_comp, full_pred),
            ssr_value if include_ssr else None, 
            weighted_ssr if include_ssr else None,
            (dpis[opt_lag:], full_pred)
        ]

        # Save reference metrics for restricted prediction
        if idx == restricted_pred_index:
            for metric in ['mape', 'mae', 'rmse', 'nrsme', 'mse', 'dtw', 'wmae', 'wmape']:
                ref_ols_performance_dct_tmp[metric.upper()] = metrics[metric]
            ref_ols_performance_dct_tmp['DTW_ALIGNED_MAPE'] = dtw_mape(actual_value_comp, restricted_pred)
            if include_ssr:
                ref_ols_performance_dct_tmp['SSR'] = np.sum((actual_value_comp - restricted_pred) ** 2)
                ref_ols_performance_dct_tmp['WEIGHTED_SSR'] = np.sum(((actual_value_comp - restricted_pred)/actual_value_comp) ** 2) / len(restricted_pred)

        # Plot if dip is in plot_dips
        if dip in plot_dips:
            granger_label = row[f'{cultivation_value_label}_granger_label']
            fig, ax = plt.subplots(figsize=(figsize))
            ax.plot(dpis, actual_value, label=f'Real {cultivation_value_label}', marker='D', color='black')
            ax.plot(dpis[opt_lag:], restricted_pred, label='Restricted\nmodel prediction', marker='o', color='#e09312', linewidth=4)
            ax.plot(dpis[opt_lag:], full_pred, label=f'Full model\nprediction with\n{dip}', marker='o', color=granger_label_color_map[granger_label], linewidth=4)
            ax.set_title(f'Prediction with Granger-{row[f"{cultivation_value_label}_granger_label"]} DIP (lag={opt_lag})')
            ax.set_xlabel(xlabel)
            ax.set_ylabel(ylabel)
            ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
            ax.set_yscale(yscale)
            plt.show()
            
    # For color correction later
    gc_prediction_summary_tmp['norm_ssr_chi2test_pval'] = gc_prediction_summary_tmp['ssr_chi2test_pval'] / 0.1
    
    # Group the DataFrame by 'granger_label'
    grouped = gc_prediction_summary_tmp.groupby('granger_label')
    gc_prediction_summary_tmp['group_norm_ssr_chi2test_pval'] = gc_prediction_summary_tmp['ssr_chi2test_pval']
    # Normalize the 'norm_ssr_chi2test' column within each group
    for label, group in grouped:
        max_value = group['group_norm_ssr_chi2test_pval'].max()  # Get the maximum value in the group
        gc_prediction_summary_tmp.loc[group.index, 'group_norm_ssr_chi2test_pval'] = group['group_norm_ssr_chi2test_pval'] / max_value  # Normalize
    
    return gc_prediction_summary_tmp, ref_ols_performance_dct_tmp

## Fitting and plotting

### all

In [117]:
plt.rcParams.update({'font.size': 16})
gc_prediction_summary = {}
ref_ols_performance_dct = {}
for fixlag in range(1, max_fixed_lag):
  print(f'Running granger prediction analysis for lag {fixlag}...')
  ref_ols_performance_dct[fixlag] = {}
  gc_prediction_summary[fixlag], ref_ols_performance_dct[fixlag] = run_granger_prediction(
      summary_df=summary_df[fixlag],
      gc_max_lag=gc_max_lag[fixlag],
      gc_test_results=test_results[fixlag],
      plot_dips=[#'PB2_129_2176', 'PB2_269_2202', 'PB2_217_2204'
                #,'PA_1989_2027','PA_172_1938', 'PA_124_1974', 'PA_142_1951'
                ],
      log_long_data=stationary_ts_df,
      restricted_pred_index=summary_df[fixlag].iloc[0].name,
      include_ssr=True,
      figsize=(5,3),
      ylabel='log10(PFU)'
  )

### corrected

In [118]:
gc_prediction_summary_corrected = {}
ref_ols_performance_corrected_dct = {}
for fixlag in range(1, max_fixed_lag):
  print(f'Running granger prediction analysis with p-value correction for lag {fixlag}...')
  gc_prediction_summary_corrected[fixlag], ref_ols_performance_corrected_dct[fixlag] = run_granger_prediction(
      summary_df=summary_df_corrected[fixlag],
      gc_max_lag=gc_max_lag[fixlag],
      gc_test_results=test_results[fixlag],
      log_long_data=stationary_ts_df,
      restricted_pred_index=summary_df_corrected[fixlag].iloc[0].name,
      include_ssr=True,
      plot_dips=[]
  )

## plot single candidates

In [119]:
def plot_single_candidate(cultivation_value_label ='plaque_assay',
                          dpis = stationary_ts_df.index.values,
                          row = summary_df[2].sort_values(by='plaque_assay_ssr_chi2test').iloc[0],
                          figsize = (7, 5),
                          xlabel = 'Time post infection (days)',
                          ylabel = 'log10(PFU)',
                          yscale = 'linear',
                          opt_lag = 2,
                          gc_prediction_summary_df = gc_prediction_summary[2],
                          gc_test_results= test_results[2],
                          log_long_data = stationary_ts_df):
  restricted_pred = gc_test_results[(cultivation_value_label, row.key)][row.plaque_assay_max_lag][1][0].predict()
  full_pred = gc_test_results[(cultivation_value_label, row.key)][row.plaque_assay_max_lag][1][1].predict()
  granger_label = row[f'{cultivation_value_label}_granger_label']
  actual_value = log_long_data[cultivation_value_label]
  dip = row.key
  print(gc_prediction_summary_df[gc_prediction_summary_df.key == dip].iloc[0]['ssr'])
  granger_label = row[f'{cultivation_value_label}_granger_label']

  fig, ax = plt.subplots(figsize=(figsize))
  ax.plot(dpis, actual_value, label=f'{cultivation_value_label}', marker='D', color='black')
  ax.plot(dpis[opt_lag:], restricted_pred, label='Restricted\nmodel prediction', marker='o', color='#e09312', linewidth=4)
  ax.plot(dpis[opt_lag:], full_pred, label=f'Full model\nprediction with\n{dip}', marker='o', color=granger_label_color_map[granger_label], linewidth=4)
  ax.set_title(f'Prediction with Granger-{row[f"{cultivation_value_label}_granger_label"]} DIP {dip.replace("_","-")} (lag={opt_lag})')
  ax.set_xlabel(xlabel)
  ax.set_ylabel(ylabel)
  ax.legend(['$\mathregular{c_{virus}}$', 'Restricted model', 'Full model'], loc='center left', bbox_to_anchor=(1, 0.5), ncol=3)
  ax.set_yscale(yscale)
  plt.show()
  
def plot_restricted_only(cultivation_value_label ='plaque_assay',
                          dpis = stationary_ts_df.index.values,
                          row = summary_df[2].sort_values(by='plaque_assay_ssr_chi2test').iloc[0],
                          figsize = (7, 5),
                          xlabel = 'Time post infection (days)',
                          ylabel = 'log10(PFU)',
                          yscale = 'linear',
                          opt_lag = 2,
                          gc_prediction_summary_df = gc_prediction_summary[2],
                          gc_test_results= test_results[2],
                          log_long_data = stationary_ts_df):
  restricted_pred = gc_test_results[(cultivation_value_label, row.key)][row.plaque_assay_max_lag][1][0].predict()
  actual_value = log_long_data[cultivation_value_label]
  dip = row.key

  fig, ax = plt.subplots(figsize=(figsize))
  ax.plot(dpis, actual_value, label=f'{cultivation_value_label}', marker='D', color='black')
  ax.plot(dpis[opt_lag:], restricted_pred, label='Restricted\nmodel prediction', marker='o', color='#e09312', linewidth=4)
  ax.set_title(f'Restricted model prediction (lag={opt_lag})')
  ax.set_xlabel(xlabel)
  ax.set_ylabel(ylabel)
  ax.legend(['$\mathregular{c_{virus}}$', 'Restricted model'], loc='center left', bbox_to_anchor=(1, 0.5), ncol=2)
  ax.set_yscale(yscale)
  plt.show()


In [120]:
for fix_lag in range(1,max_fixed_lag):
  plot_restricted_only(cultivation_value_label='plaque_assay',
                        row=summary_df[fix_lag].sort_values(by='plaque_assay_ssr_chi2test').iloc[0],
                        figsize=(7, 5),
                        xlabel='Time post infection (days)',
                        ylabel='log10(PFU)',
                        yscale='linear',
                        gc_prediction_summary_df=gc_prediction_summary[fix_lag],
                        gc_test_results= test_results[fix_lag],
                        opt_lag=fix_lag)

# Swarmplots

In [121]:
import seaborn as sns

## Helper Functions

In [122]:
def get_pval_symbol(pval):
    if pval < 0.0001:
        return '****'
    elif pval < 0.001:
        return '***'
    elif pval < 0.01:
        return '**'
    elif pval < 0.05:
        return '*'
    else:
        return ''

def better_percentage(group, val):
    n = len(group)
    perc = round(len(group[group < val]) / n * 100)
    return f'{perc}%\nbetter'

def get_main_color(granger_label):
    if granger_label == 'caused':
        return 'tab:red'
    elif granger_label == 'bi-directional':
        return 'tab:purple'
    elif granger_label == 'causing':
        return 'tab:blue'
    else:
        return 'grey'

def get_corrected_color(row, column='granger_score', invert=True):
    if row['granger_label'] == 'caused':
        base_color = plt.get_cmap('Reds')
    elif row['granger_label'] == 'bi-directional':
        base_color = plt.get_cmap('Purples')
    elif row['granger_label'] == 'causing':
        base_color = plt.get_cmap('Blues')
    else:
        base_color = plt.get_cmap('Greys')

    if invert == True:
        base_color = base_color.reversed()
        
    return base_color(row[column])

# Return a lighter shade based on the granger_score (the lower the score, the lighter the shade)
def get_corrected_color_v2(row, label_column, categories=[None,None,None], column='granger_score', invert=True):
    if row[label_column] == categories[0]:
        base_color = plt.get_cmap('Reds')
    elif row[label_column] == categories[1]:
        base_color = plt.get_cmap('Purples')
    elif row[label_column] == categories[2]:
        base_color = plt.get_cmap('Blues')
    else:
        base_color = plt.get_cmap('Greys')

    if invert == False:
        base_color = base_color.reversed()
        
    # Return a lighter shade based on the granger_score (the lower the score, the lighter the shade)
    return base_color(row[column])

# Function to calculate Cohen's d
def cohens_d(group1, group2):
    # Calculate means and standard deviations
    mean1, mean2 = np.mean(group1), np.mean(group2)
    std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((len(group1) - 1) * std1 ** 2 + (len(group2) - 1) * std2 ** 2) / (len(group1) + len(group2) - 2))
    
    # Cohen's d
    return (mean1 - mean2) / pooled_std

# Function to calculate Cliff's delta
def cliffs_delta(group1, group2):
    m, n = len(group1), len(group2)
    count = sum(1 if x > y else -1 if x < y else 0 for x in group1 for y in group2)
    return count / (m * n)

def test_statistical_difference(group1, group2, alpha=0.05):
    """
    Tests if two groups are statistically different from each other and returns effect size.

    Parameters:
    - group1, group2: The two groups to be compared.
    - alpha: Significance level for the tests.

    Returns:
    - A tuple containing:
    - p-value from the test
    - Effect size (Cohen's d for t-test, Cliff's delta for Mann-Whitney U test)
    - Test type ('t-test' or 'mann-whitney')
    """
    
    # Test for normality
    shapiro_test_group1 = shapiro(group1)
    shapiro_test_group2 = shapiro(group2)
    
    if shapiro_test_group1.pvalue > alpha and shapiro_test_group2.pvalue > alpha:
        # If both groups are normal, use T-test
        t_stat, t_pvalue = ttest_ind(list(group1), list(group2))
        
        # Calculate Cohen's d
        d = cohens_d(group1, group2)
        
        return t_pvalue, d, 't-test'
    else:
        # If either group is not normal, use Mann-Whitney U test
        mw_stat, mw_pvalue = mannwhitneyu(group1, group2)
        
        # Calculate Cliff's delta
        delta = cliffs_delta(group1, group2)
        
        return mw_pvalue, delta, 'mann-whitney'



## Functions

In [123]:
effect_size_dct = {'mann-whitney': 'Cliff\'s delta', 't-test': 'Cohen\'s d'}

def make_performance_swarmplot(dip_forecast_summary,
                                ref_performance_dct,
                                performance_metric='mape',
                                forecasted_value = 'pfu',
                                title = 'Granger-related DI vRNAs predictive power over PFU\n',
                                ylabel='Mean Absolute Percentage Error (MAPE)',
                                dot_color_ref = 'diff_level_norm',
                                inverted_colors = True, # the higher the value, the darker the color
                                upper_border = -10,
                                ymax = None,
                                ymin = 0,
                                yscale = 'linear',
                                p_val_pos = 0,
                                figsize=(8, 4),
                                better_perc = False,
                                markersize=18,
                                linewidth=4,
                                dotsize=5,
                                included_dvgs = None
                                ): 
    dip_forecast_summary['corrected_color'] = dip_forecast_summary.apply(get_corrected_color, args=(dot_color_ref, inverted_colors), axis=1)
    
    if included_dvgs is not None:
        dip_forecast_summary = dip_forecast_summary[dip_forecast_summary.key.isin(included_dvgs)]
        
    # Create the swarm plot
    fig, axs = plt.subplots(nrows=1, figsize=figsize)
    ax = axs
    ax = sns.swarmplot(x='granger_label', 
                        y=performance_metric, 
                        hue='key',
                        data=dip_forecast_summary, 
                        order = ['causing', 'bi-directional', 'caused', 'non-related'],
                        palette=dip_forecast_summary['corrected_color'].tolist(),
                        size=dotsize, ax=ax, legend=False, linewidth=0.2
    )
    # Set title and other plot parameters
    ax.set_title(title, pad=15)
    ax.set_xlabel('Granger causality label', labelpad=10)

    ax.set_ylabel(ylabel)
    xmin, xmax = ax.get_xlim()
    ax.hlines(y=ref_performance_dct[performance_metric.upper()], xmin=xmin, xmax=xmax, label='restricted model performance', color='#e09312', linewidth=linewidth, zorder=10)
    ax.legend(loc='lower right')
    if ymax is None:
        ymax = ref_performance_dct[performance_metric.upper()]*3
    ax.set_ylim(ymin, ymax)
    ax.set_yscale(yscale)
    #ax.invert_yaxis()
    ax.tick_params(top=False, labeltop=False, bottom=True, labelbottom=True)

    xs, xticklabels = ax.get_xticks(), ax.get_xticklabels()

    # Loop through the x-axis categories and plot the medians as markers
    for x, xticklabel in zip(xs, xticklabels):
        label = xticklabel.get_text()
        
        # Calculate median
        median_value = dip_forecast_summary[dip_forecast_summary.granger_label == label][performance_metric].median()
        
        # Plot the median as a horizontal marker (-)
        ax.plot(x, median_value, marker='+', color='black', markersize=markersize, zorder=11, label=f'{label} Median')
        
        ax.text(x, p_val_pos, f'n={len(dip_forecast_summary[dip_forecast_summary.granger_label == label])}', ha='center', va='bottom', color='black', zorder=10)

        if better_perc:
            ax.text(x, upper_border, 
                better_percentage(dip_forecast_summary[dip_forecast_summary.granger_label == label][performance_metric], 
                                        ref_performance_dct[performance_metric.upper()]), 
                    ha='center', va='bottom', color='black', zorder=10)
    
    fig.tight_layout()
    return fig, ax

## all 

In [124]:
plt.rcParams.update({'font.size': 28})
! mkdir -p {output_prefix}/plots


In [125]:
label_order = ['causing', 'bi-directional', 'caused', 'non-related']
for fixlag in range(1, max_fixed_lag):
  gc_prediction_summary[fixlag] = gc_prediction_summary[fixlag].sort_values(by='granger_label', key=lambda x: x.map({label: i for i, label in enumerate(label_order)}))
  gc_prediction_summary_corrected[fixlag] = gc_prediction_summary_corrected[fixlag].sort_values(by='granger_label', key=lambda x: x.map({label: i for i, label in enumerate(label_order)}))

In [128]:
plt.rcParams.update({'font.size': 28})
for fixlag in range(1, max_fixed_lag):
  gc_prediction_summary[fixlag]['non_zero_counts'] = gc_prediction_summary[fixlag]['key'].apply(lambda x: dvg2nonzero_counts[x])
  print(f'Creating performance swarmplot for lag {fixlag}...')
  fig, ax = make_performance_swarmplot(
      dip_forecast_summary=gc_prediction_summary[fixlag],
      ref_performance_dct=ref_ols_performance_dct[fixlag],
      performance_metric='ssr',
      forecasted_value = 'plaque_assay',
      title = f'Shuffled Granger-related DVGs predictive power over PFU (lag={fixlag})',
      ylabel='SSR',
      dot_color_ref = 'non_zero_counts',
      inverted_colors=True,
      upper_border=-0.05,
      ymax=ref_ols_performance_dct[fixlag]['SSR']*2,
      ymin=-10,
      p_val_pos=ref_ols_performance_dct[fixlag]['SSR']*1.5,
      figsize=(18,7),
      dotsize=8,
      linewidth=5,
      included_dvgs=ts_data_shuffled.columns.tolist()
  )
  fig.savefig(f'{output_prefix}/plots/shuffled_ssr_swarmplot_lag{fixlag}.png')

In [129]:
plt.rcParams.update({'font.size': 28})
for fixlag in range(1, max_fixed_lag):
  gc_prediction_summary_corrected[fixlag]['non_zero_counts'] = gc_prediction_summary_corrected[fixlag]['key'].apply(lambda x: dvg2nonzero_counts[x])
  print(f'Creating performance swarmplot for lag {fixlag}...')
  fig, ax = make_performance_swarmplot(
      dip_forecast_summary=gc_prediction_summary[fixlag],
      ref_performance_dct=ref_ols_performance_dct[fixlag],
      performance_metric='ssr',
      forecasted_value = 'plaque_assay',
      title = f'Random Granger-related DVGs predictive power over PFU (lag={fixlag})',
      ylabel='SSR',
      dot_color_ref = 'non_zero_counts',
      inverted_colors=True,
      upper_border=-0.05,
      ymax=ref_ols_performance_dct[fixlag]['SSR']*2,
      ymin=-10,
      p_val_pos=ref_ols_performance_dct[fixlag]['SSR']*1.5,
      figsize=(18,7),
      dotsize=8,
      linewidth=5,
      included_dvgs=ts_data_random.columns.tolist()
  )
  fig.savefig(f'{output_prefix}/plots/random_ssr_swarmplot_lag{fixlag}.png')

## corrected

In [130]:

for fix_lag in range(1,max_fixed_lag):
  gc_prediction_summary_corrected[fix_lag]['non_zero_counts'] = gc_prediction_summary_corrected[fix_lag]['key'].apply(lambda x: dvg2nonzero_counts[x])
  fig, ax = make_performance_swarmplot(gc_prediction_summary_corrected[fix_lag],
                              ref_ols_performance_corrected_dct[fix_lag],
                              performance_metric='ssr',
                              forecasted_value = 'plaque_assay',
                              title = f'Random Granger-related DI vRNAs predictive power over PFU with p-value correction (lag={fix_lag})',
                              ylabel='SSR',
                              dot_color_ref = 'non_zero_counts',
                              inverted_colors=True,
                              upper_border=-0.05,
                              ymax=ref_ols_performance_corrected_dct[fix_lag]['SSR']*2,
                              ymin=-10,
                              p_val_pos=ref_ols_performance_corrected_dct[fix_lag]['SSR']*1.5,
                              figsize=(18,7),
                              dotsize=8,
                              linewidth=5,
                              included_dvgs=ts_data_random.columns.tolist()
                              )
  fig.savefig(f'{output_prefix}/plots/random_ssr_swarmplot_corrected_lag{fix_lag}.png')
  fig.show()

In [131]:

for fix_lag in range(1, max_fixed_lag):
  gc_prediction_summary_corrected[fix_lag]['non_zero_counts'] = gc_prediction_summary_corrected[fix_lag]['key'].apply(lambda x: dvg2nonzero_counts[x])
  fig, ax = make_performance_swarmplot(gc_prediction_summary_corrected[fix_lag],
                              ref_ols_performance_corrected_dct[fix_lag],
                              performance_metric='ssr',
                              forecasted_value = 'plaque_assay',
                              title = f'Shuffled Granger-related DI vRNAs predictive power over PFU with p-value correction (lag={fix_lag})',
                              ylabel='SSR',
                              dot_color_ref = 'non_zero_counts',
                              inverted_colors=True,
                              upper_border=-0.05,
                              ymax=ref_ols_performance_corrected_dct[fix_lag]['SSR']*2,
                              ymin=-10,
                              p_val_pos=ref_ols_performance_corrected_dct[fix_lag]['SSR']*1.5,
                              figsize=(18,7),
                              dotsize=8,
                              linewidth=5,
                              included_dvgs=ts_data_shuffled.columns.tolist()
                              )
  fig.savefig(f'{output_prefix}/plots/shuffled_ssr_swarmplot_corrected_lag{fix_lag}.png')
  fig.show()

# group sizes with increasing lag

## all

In [444]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 3))

for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  group_sizes = []    
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary[fixlag][gc_prediction_summary[fixlag]['granger_label'] == label]
    group_sizes.append(len(subset))
  ax.plot(range(1, max_fixed_lag), group_sizes, marker='o', label=label, color=color)
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Number of DVGs')
ax.set_title('Number of Granger-related DVGs across different fixed lags')
# Add legend next to the plot
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_xticks(range(1, max_fixed_lag))
plt.tight_layout()

## corrected

In [446]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 3))

for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  group_sizes = []    
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary_corrected[fixlag][gc_prediction_summary_corrected[fixlag]['granger_label'] == label]
    group_sizes.append(len(subset))
  ax.plot(range(1, max_fixed_lag), group_sizes, marker='o', label=label, color=color)
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Number of DVGs')
ax.set_title('Number of Granger-related DVGs across different fixed lags (with p-value correction)')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_xticks(range(1, max_fixed_lag))
plt.tight_layout()

## all vs corrected

In [74]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 3))

for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  group_sizes = []
  group_sizes_corrected = [] 
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary[fixlag][gc_prediction_summary[fixlag]['granger_label'] == label]
    group_sizes.append(len(subset))
    
    subset_corrected = gc_prediction_summary_corrected[fixlag][gc_prediction_summary_corrected[fixlag]['granger_label'] == label]
    group_sizes_corrected.append(len(subset_corrected))
  ax.plot(range(1, max_fixed_lag), group_sizes, marker='o', label=label, color=color, linestyle='solid')
  ax.plot(range(1, max_fixed_lag), group_sizes_corrected, marker='D', label=f'{label} (corrected)', color=color, linestyle='dashed')
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Number of DVGs')
ax.set_title('Number of Granger-related DVGs across different fixed lags')
# Add legend next to the plot
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_xticks(range(1, max_fixed_lag))
plt.tight_layout()

# SSR with increasing lag

In [430]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 3))
for fixlag in range(1, max_fixed_lag):
    ax.scatter(fixlag,
              ref_ols_performance_dct[fixlag]['SSR'],
              color='tab:orange',
              label='restricted model' if fixlag == 1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Restricted model SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Restricted model SSR vs. fixed lag used in Granger causality test')

mean_ssr_full_model = []
for fixlag in range(1, max_fixed_lag):
    mean_ssr_full_model.append(gc_prediction_summary[fixlag]['ssr'].median())
    ax.scatter(fixlag,
              gc_prediction_summary[fixlag]['ssr'].median(),
              color='tab:blue',
              label='full model' if fixlag==1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Median SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Median SSR vs. fixed lag used in Granger causality test')
ax.legend(loc='upper right')
plt.show()

In [432]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 5))
for fixlag in range(1, max_fixed_lag):
    ax.scatter(fixlag,
              ref_ols_performance_dct[fixlag]['SSR'],
              color='tab:orange',
              label='restricted model' if fixlag == 1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Restricted model SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Restricted model SSR vs. fixed lag used in Granger causality test')

mean_ssr_full_model = []


for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  mean_ssr_full_model = []    
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary[fixlag][gc_prediction_summary[fixlag]['granger_label'] == label]
    mean_ssr_full_model.append(subset['ssr'].median())
    ax.scatter(fixlag,
              subset['ssr'].median(),
              color=color,
              label=label if fixlag==1 else "",
              linewidth=2)
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Median SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Median SSR vs. fixed lag used in Granger causality test')
plt.show()

## retransformed

In [433]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 3))
for fixlag in range(1, max_fixed_lag):
    ax.scatter(fixlag,
              ref_ols_performance_retransformed_dct[fixlag]['SSR'],
              color='tab:orange',
              label='restricted model' if fixlag == 1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Restricted model SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Restricted model SSR vs. fixed lag used in Granger causality test')

mean_ssr_full_model = []
for fixlag in range(1, max_fixed_lag):
    mean_ssr_full_model.append(gc_prediction_summary_retransformed[fixlag]['ssr'].median())
    ax.scatter(fixlag,
              gc_prediction_summary_retransformed[fixlag]['ssr'].median(),
              color='tab:blue',
              label='full model' if fixlag==1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Median SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Median SSR vs. fixed lag used in Granger causality test')
ax.legend(loc='upper right')
plt.show()

In [435]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 5))
for fixlag in range(1, max_fixed_lag):
    ax.scatter(fixlag,
              ref_ols_performance_retransformed_dct[fixlag]['SSR'],
              color='tab:orange',
              label='restricted model' if fixlag == 1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Restricted model SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Restricted model SSR vs. fixed lag used in Granger causality test')

mean_ssr_full_model = []


for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  mean_ssr_full_model = []    
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary_retransformed[fixlag][gc_prediction_summary_retransformed[fixlag]['granger_label'] == label]
    mean_ssr_full_model.append(subset['ssr'].median())
    ax.scatter(fixlag,
              subset['ssr'].median(),
              color=color,
              label=label if fixlag==1 else "",
              linewidth=2)
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Median SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Median SSR vs. fixed lag used in Granger causality test')
plt.show()

## corrected

In [436]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 3))
for fixlag in range(1, max_fixed_lag):
    ax.scatter(fixlag,
              ref_ols_performance_corrected_dct[fixlag]['SSR'],
              color='tab:orange',
              label='restricted model' if fixlag == 1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Restricted model SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Restricted model SSR vs. fixed lag used in Granger causality test')

mean_ssr_full_model = []
for fixlag in range(1, max_fixed_lag):
    mean_ssr_full_model.append(gc_prediction_summary_corrected[fixlag]['ssr'].median())
    ax.scatter(fixlag,
              gc_prediction_summary_corrected[fixlag]['ssr'].median(),
              color='tab:blue',
              label='full model' if fixlag==1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Median SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Median SSR vs. fixed lag used in Granger causality test')
ax.legend(loc='upper right')
plt.show()

In [438]:
plt.rcParams.update({'font.size': 10})
fig, ax = plt.subplots(figsize=(5, 5))
for fixlag in range(1, max_fixed_lag):
    ax.scatter(fixlag,
              ref_ols_performance_corrected_dct[fixlag]['SSR'],
              color='tab:orange',
              label='restricted model' if fixlag == 1 else "")
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Restricted model SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Restricted model SSR vs. fixed lag used in Granger causality test')

mean_ssr_full_model = []


for label, color in zip(['causing', 'bi-directional', 'caused', 'non-related'],
                        ['tab:blue', 'tab:purple', 'tab:red', 'gray']):
  mean_ssr_full_model = []    
  for fixlag in range(1, max_fixed_lag):
    subset = gc_prediction_summary_corrected[fixlag][gc_prediction_summary_corrected[fixlag]['granger_label'] == label]
    mean_ssr_full_model.append(subset['ssr'].median())
    ax.scatter(fixlag,
              subset['ssr'].median(),
              color=color,
              label=label if fixlag==1 else "",
              linewidth=2)
ax.set_xlabel('Fixed lag used in Granger causality test')
ax.set_ylabel('Median SSR')
ax.set_xticks(range(1, max_fixed_lag))
ax.set_title('Median SSR vs. fixed lag used in Granger causality test')
ax.legend(loc='upper right')
plt.show()